In [1]:
import pandas as pd 
import matplotlib.pyplot as plt 
from pathlib import Path
import numpy as np

In [2]:
def resolve_project_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in (path, path.parent):
        if (candidate / "data" / "raw_data").exists():
            return candidate
    raise FileNotFoundError("Could not find data/raw_data.")

ROOT = resolve_project_root()
raw_le_dir = ROOT / "data" / "raw_data" / "life_expectancy_total"
data_path = raw_le_dir / "API_SP.DYN.LE00.IN_DS2_en_csv_v2_399797.csv"
df = pd.read_csv(data_path, skiprows=4, header=0)

df = df.rename(columns={
"Country Name": "country_name",
"Country Code": "country_code",
"Indicator Name": "indicator_name",
"Indicator Code": "indicator_code"
})


year_cols = [col for col in df.columns if str(col).isdigit()]

df_long = df.melt(
id_vars=["country_name", "country_code", "indicator_name", "indicator_code"],
value_vars=year_cols,
var_name="Year",
value_name="life_expectancy_total"
)

df_long["Year"] = df_long["Year"].astype(int)
df_long["life_expectancy_total"] = pd.to_numeric(df_long["life_expectancy_total"], errors="coerce")
df_long = df_long.dropna(subset=["life_expectancy_total"]).reset_index(drop=True)




df_long.head()

,country_name,country_code,indicator_name,indicator_code,Year,life_expectancy_total
0,Aruba,ABW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,1960,64.049000
1,Africa Eastern and Southern,AFE,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,1960,44.169658
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,1960,32.799000
3,Africa Western and Central,AFW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,1960,37.779636
4,Angola,AGO,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,1960,37.933000


In [3]:

meta_path = raw_le_dir / "Metadata_Country_API_SP.DYN.LE00.IN_DS2_en_csv_v2_399797.csv"



# Read country metadata
meta = pd.read_csv(meta_path, header=0)
meta = meta.rename(columns={
"Country Code": "country_code",
"IncomeGroup": "income_group",
"SpecialNotes": "special_notes",
"TableName": "table_name",
})

In [4]:
meta

,country_code,Region,income_group,special_notes,table_name,Unnamed: 5
0,ABW,Latin America & Caribbean,High income,NaN,Aruba,NaN
1,AFE,NaN,NaN,"26 countries, stretching from the Red Sea in t...",Africa Eastern and Southern,NaN
2,AFG,Middle East & North Africa,Low income,The reporting period for national accounts dat...,Afghanistan,NaN
3,AFW,NaN,NaN,"22 countries, stretching from the westernmost ...",Africa Western and Central,NaN
4,AGO,Sub-Saharan Africa,Lower middle income,The World Bank systematically assesses the app...,Angola,NaN
...,...,...,...,...,...,...
259,XKX,Europe & Central Asia,Upper middle income,NaN,Kosovo,NaN
260,YEM,Middle East & North Africa,Low income,The World Bank systematically assesses the app...,"Yemen, Rep.",NaN
261,ZAF,Sub-Saharan Africa,Upper middle income,Fiscal year end: March 31; reporting period fo...,South Africa,NaN
262,ZMB,Sub-Saharan Africa,Lower middle income,National accounts data were rebased to reflect...,Zambia,NaN


In [5]:
df_merged = df_long.merge(meta, on = 'country_code', how = 'left')

In [6]:
df_merged.head()

,country_name,country_code,indicator_name,indicator_code,Year,life_expectancy_total,Region,income_group,special_notes,table_name,Unnamed: 5
0,Aruba,ABW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,1960,64.049000,Latin America & Caribbean,High income,NaN,Aruba,NaN
1,Africa Eastern and Southern,AFE,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,1960,44.169658,NaN,NaN,"26 countries, stretching from the Red Sea in t...",Africa Eastern and Southern,NaN
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,1960,32.799000,Middle East & North Africa,Low income,The reporting period for national accounts dat...,Afghanistan,NaN
3,Africa Western and Central,AFW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,1960,37.779636,NaN,NaN,"22 countries, stretching from the westernmost ...",Africa Western and Central,NaN
4,Angola,AGO,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,1960,37.933000,Sub-Saharan Africa,Lower middle income,The World Bank systematically assesses the app...,Angola,NaN
